# Experimentation: Tuning BM25 and BM25+Model1 on StackOverflow

This notebook mirrors the BM25/Model1 tuning flow from the demo notebook, but targets a StackOverflow fusion collection built from your split-specific collections.

It assumes you already trained IBM Model 1 for the field `text` in `stackoverflow_tran`, and that you will run tuning on a combined collection containing both `train` and `dev1` input data.

Trained on tran !

### Setup
1. Set `COLLECT_ROOT`.
2. Move to `flexneuart_scripts`.
3. Choose or create a combined fusion collection that contains both `input_data/train` and `input_data/dev1`.
4. Set `TRAIN_CAND_QTY` for fusion fine-tuning.

In [6]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path('/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP')
SCRIPTS_DIR = REPO_ROOT / 'demo' / 'flexneuart_scripts'
COLLECT_ROOT = (REPO_ROOT / 'demo' / 'collections').resolve()
# This must be a collection that contains both `input_data/train` and `input_data/dev1`.
COLLECT_NAME = 'stackoverflow'

# Train the fusion model on `train` and evaluate it on `dev1`.
TRAIN_PART = 'train'
TEST_PART = 'dev1'
TRAIN_CAND_QTY = 15

# Field choices for StackOverflow conversion notebook: text, text_unlemm, text_raw, bigram
BM25_INDEX_FIELD = 'text'
BM25_QUERY_FIELD = 'text'
MODEL1_INDEX_FIELD = 'text'
MODEL1_QUERY_FIELD = 'text'

os.environ['COLLECT_ROOT'] = str(COLLECT_ROOT)

collect_dir = COLLECT_ROOT / COLLECT_NAME
exper_desc_dir = collect_dir / 'exper_desc'
exper_desc_dir.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT    =', REPO_ROOT)
print('SCRIPTS_DIR  =', SCRIPTS_DIR)
print('COLLECT_ROOT =', COLLECT_ROOT)
print('COLLECT_NAME =', COLLECT_NAME)
print('TRAIN_PART   =', TRAIN_PART)
print('TEST_PART    =', TEST_PART)
print('TRAIN_CAND_QTY =', TRAIN_CAND_QTY)
print('EXPER_DESC   =', exper_desc_dir)

required_paths = [
    collect_dir / 'input_data' / TRAIN_PART / 'QuestionFields.jsonl',
    collect_dir / 'input_data' / TRAIN_PART / 'AnswerFields.jsonl',
    collect_dir / 'input_data' / TRAIN_PART / 'qrels.txt',
    collect_dir / 'input_data' / TEST_PART / 'QuestionFields.jsonl',
    collect_dir / 'input_data' / TEST_PART / 'AnswerFields.jsonl',
    collect_dir / 'input_data' / TEST_PART / 'qrels.txt',
    collect_dir / 'lucene_index',
    collect_dir / 'forward_index' / BM25_INDEX_FIELD,
    collect_dir / 'derived_data' / 'giza' / f'{MODEL1_INDEX_FIELD}' / 'output.t1.5.bin'
]

for p in required_paths:
    if not p.exists():
        raise FileNotFoundError(f'Missing prerequisite: {p}')

print('All prerequisites found.')

REPO_ROOT    = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP
SCRIPTS_DIR  = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/flexneuart_scripts
COLLECT_ROOT = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections
COLLECT_NAME = stackoverflow
TRAIN_PART   = train
TEST_PART    = dev1
TRAIN_CAND_QTY = 15
EXPER_DESC   = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow/exper_desc
All prerequisites found.


## Tuning BM25
Generate BM25 tuning descriptors for StackOverflow (`text` vs `text`).

In [7]:
cmd = [
    'python3', './gen_exper_desc/gen_bm25_tune_json_desc.py',
    '--index_field_name', BM25_INDEX_FIELD,
    '--query_field_name', BM25_QUERY_FIELD,
    '--outdir', str(exper_desc_dir),
    '--exper_subdir', 'tuning',
    '--rel_desc_path', 'exper_desc'
]
print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'BM25 descriptor generation failed: {res.returncode}')

Running: python3 ./gen_exper_desc/gen_bm25_tune_json_desc.py --index_field_name text --query_field_name text --outdir /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow/exper_desc --exper_subdir tuning --rel_desc_path exper_desc
Namespace(outdir='/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow/exper_desc', rel_desc_path='exper_desc', exper_subdir='tuning', index_field_name='text', query_field_name='text', cand_prov_uri=None, cand_prov_add_conf=None, cand_prov_qty=None)



In [8]:
bm25_desc = f'exper_desc/bm25tune_{BM25_QUERY_FIELD}_{BM25_INDEX_FIELD}.json'
cmd = [
    'bash', './exper/run_experiments.sh',
    COLLECT_NAME,
    bm25_desc,
    '-test_part', TEST_PART,
    '-train_part', TRAIN_PART,
    '-train_cand_qty', str(TRAIN_CAND_QTY)
]
print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'BM25 tuning experiments failed: {res.returncode}')

Running: bash ./exper/run_experiments.sh stackoverflow exper_desc/bm25tune_text_text.json -test_part dev1 -train_part train -train_cand_qty 15
Using collection root: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections
The number of CPU cores:      16
The number of || experiments: 1
The number of threads:        16
Experiment descriptor file:                                 /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow/exper_desc/bm25tune_text_text.json
Default test set:                                           dev1
Number of parallel experiments:                             1
Number of threads in feature extractors/query applications: 16
Parsed experiment parameters:
experSubdir:tuning/bm25tune_text_text/bm25tune_k1=0.4_b=0.3
extrTypeFinal:exper_desc/bm25tune_text_text/bm25tune_k1=0.4_b=0.3.json
testOnly:1
modelFinal:exper_desc/models/one_feat.model
Started a process 1149602, working di

In [9]:
bm25_desc = f'exper_desc/bm25tune_{BM25_QUERY_FIELD}_{BM25_INDEX_FIELD}.json'
cmd = [
    'bash', './report/get_exper_results.sh',
    COLLECT_NAME,
    bm25_desc,
    'bm25tune_stackoverflow.tsv',
    '-test_part', TEST_PART,
    '-flt_cand_qty', '250',
    '-print_best_metr', 'map'
]
print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'BM25 results extraction failed: {res.returncode}')

Running: bash ./report/get_exper_results.sh stackoverflow exper_desc/bm25tune_text_text.json bm25tune_stackoverflow.tsv -test_part dev1 -flt_cand_qty 250 -print_best_metr map
Using collection root: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections
Including only runs that generated 250 candidate records
Best results for metric map:
Value: 0.000000
Result sub-dir: tuning/bm25tune_text_text/bm25tune_k1=0.4_b=0.3



## Tuning a Fusion of Model 1 and BM25
Set `-k1` and `-b` below to the best BM25 values from the previous step.

In [ ]:
# Replace with best BM25 params from bm25tune_stackoverflow.tsv
BEST_BM25_K1 = 0.6
BEST_BM25_B = 0.3

cmd = [
    'python3', './gen_exper_desc/gen_model1_exper_json_desc.py',
    '-k1', str(BEST_BM25_K1),
    '-b', str(BEST_BM25_B),
    '--exper_subdir', 'tuning',
    '--query_field_name', MODEL1_QUERY_FIELD,
    '--index_field_name', MODEL1_INDEX_FIELD,
    '--outdir', str(exper_desc_dir),
    '--rel_desc_path', 'exper_desc'
]
print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'Model1+BM25 descriptor generation failed: {res.returncode}')

In [ ]:
model1_desc = f'exper_desc/model1tune_{MODEL1_QUERY_FIELD}_{MODEL1_INDEX_FIELD}.json'
cmd = [
    'bash', './exper/run_experiments.sh',
    COLLECT_NAME,
    model1_desc,
    '-test_part', TEST_PART,
    '-train_part', TRAIN_PART,
    '-train_cand_qty', str(TRAIN_CAND_QTY)
]
print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'Model1+BM25 tuning experiments failed: {res.returncode}')

In [ ]:
model1_desc = f'exper_desc/model1tune_{MODEL1_QUERY_FIELD}_{MODEL1_INDEX_FIELD}.json'
cmd = [
    'bash', './report/get_exper_results.sh',
    COLLECT_NAME,
    model1_desc,
    'model1tune_stackoverflow.tsv',
    '-test_part', TEST_PART,
    '-flt_cand_qty', '250',
    '-print_best_metr', 'map'
]
print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'Model1+BM25 results extraction failed: {res.returncode}')

### Notes
- `TRAIN_PART` is the split used to fit the fusion setup; `TEST_PART` is the split used to evaluate it.
- The collection passed to the scripts must already contain both split directories. If it does not, you need to create a combined fusion collection first.
- `TRAIN_CAND_QTY = 15` means each training query keeps the top 15 BM25 candidates for Model 1 + BM25 fine-tuning.
- Output summaries are written to `bm25tune_stackoverflow.tsv` and `model1tune_stackoverflow.tsv` in `demo/flexneuart_scripts`.